# Job Transition Analytics Project

This project demonstrates data analysis and predictive modeling skills using a synthetic dataset of employees transitioning between roles. The goal is to explore factors that influence the success of moving into target roles (Business Analyst, Program Manager, Data Analyst).

The notebook includes:
- Data loading and inspection
- Exploratory data analysis with visualizations
- Data preprocessing (encoding categorical variables and splitting datasets)
- Building and evaluating predictive models (Logistic Regression, Decision Tree, Random Forest)
- Discussion of feature importance and insights


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('synthetic_job_transition.csv')
df.head()

## Data Overview

Below we inspect the basic structure of the dataset, including column types and summary statistics.

In [ ]:
# Display basic information and summary statistics
df.info()
df.describe(include='all').T

## Exploratory Data Analysis

### Distribution of Numeric Features

In [ ]:
numeric_cols = ['years_experience','performance_rating','satisfaction_level','training_hours',
                'certifications_count','last_promotion_years','projects_completed']
fig, axes = plt.subplots(len(numeric_cols), 1, figsize=(8, 24))
for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

### Target Variable Distribution

In [ ]:
sns.countplot(x='target_role_success', data=df)
plt.title('Distribution of Target Role Success')
plt.show()

### Correlation Matrix (Numeric Features)

In [ ]:
corr = df[numeric_cols + ['target_role_success']].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

## Predictive Modeling

We will build three classification models to predict whether an employee successfully transitions into their target role:

1. **Logistic Regression**
2. **Decision Tree Classifier**
3. **Random Forest Classifier**

We will encode categorical variables using one-hot encoding and evaluate the models using accuracy and confusion matrices.

In [ ]:
# Separate features and target
X = df.drop(columns=['target_role_success', 'employee_id'])
y = df['target_role_success']

# Identify categorical and numeric columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# Preprocess data: OneHotEncode categorical variables
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

### Logistic Regression

In [ ]:
log_reg = Pipeline(steps=[('preprocessor', preprocessor),
                          ('classifier', LogisticRegression(max_iter=1000))])

log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print('Logistic Regression Accuracy:', acc_lr)
print('Confusion Matrix:
', confusion_matrix(y_test, y_pred_lr))
print('Classification Report:
', classification_report(y_test, y_pred_lr))

### Decision Tree Classifier

In [ ]:
dt = Pipeline(steps=[('preprocessor', preprocessor),
                    ('classifier', DecisionTreeClassifier(max_depth=5, random_state=42))])
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)
print('Decision Tree Accuracy:', acc_dt)
print('Confusion Matrix:
', confusion_matrix(y_test, y_pred_dt))
print('Classification Report:
', classification_report(y_test, y_pred_dt))

### Random Forest Classifier

In [ ]:
rf = Pipeline(steps=[('preprocessor', preprocessor),
                    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))])
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print('Random Forest Accuracy:', acc_rf)
print('Confusion Matrix:
', confusion_matrix(y_test, y_pred_rf))
print('Classification Report:
', classification_report(y_test, y_pred_rf))

# Feature importance
# After preprocessing, retrieve feature names
ohe = rf.named_steps['preprocessor'].transformers_[0][1]
cat_feature_names = ohe.get_feature_names_out(categorical_cols)
feature_names = list(cat_feature_names) + numeric_cols
importances = rf.named_steps['classifier'].feature_importances_
feature_importance = pd.Series(importances, index=feature_names).sort_values(ascending=False)
feature_importance.head(10)

## Conclusions

In this project we generated a synthetic dataset representing employees attempting to transition to new roles. We explored the distribution of various features and built three machine learning models to predict the success of these transitions.

Key observations:
- Features like performance rating, satisfaction level, and certifications had strong positive correlations with success.
- The Random Forest model achieved the highest accuracy among the tested models, suggesting nonlinear relationships between the features and the target variable.
- Feature importance analysis shows which factors are most influential in predicting successful role transitions.

This analysis demonstrates the ability to create a dataset, perform comprehensive EDA, and build predictive models suitable for business analytics and data science roles.